In [1]:
#Import basic packages
import numpy as np
import pandas as pd
import csv

import os
import argparse

#Import keras modules
import tensorflow as tf
import keras.backend as K
#import keras.backend.tensorflow_backend as KTF
import keras
import keras.layers
from keras.layers import Layer 
import keras.initializers
from keras.models import Model, Sequential,load_model
from keras.layers import Input, Dense, Dropout, BatchNormalization, Activation, Multiply, multiply,dot
from keras.layers import Concatenate,concatenate
from keras.optimizers import Adam
from keras.utils import plot_model

#Fix the random seed
np.random.seed(5)

2026-01-28 21:43:30.482717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769633010.636938  430959 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769633010.676803  430959 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769633010.952194  430959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769633010.952241  430959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769633010.952243  430959 computation_placer.cc:177] computation placer alr

# CV evaluation

In [2]:
gdsc_gex = pd.read_csv(
        "../../msigdb_GEX_data_filtered_logCPM.csv",
        index_col=0
    )

In [3]:
pdo_gex = pd.read_csv(
    "../../integrated_data/pdo_logCPM.csv",
    index_col=0
)

pdo_dr = pd.read_csv(
    "../../integrated_data/dose_response_pdo.csv"
)

smiles = pd.read_csv(
    "../../integrated_data/vector_smiles.csv",
    index_col=0
)

In [4]:
pdo_gex = pdo_gex.loc[
    :,pdo_gex.columns.intersection(gdsc_gex.columns)]
pdo_gex = pdo_gex.reindex(columns=gdsc_gex.columns, fill_value=0.0)
pdo_gex.shape

(54, 4377)

In [5]:
pdo_dr = pdo_dr.loc[pdo_dr.seen_before=="yes",:]

In [6]:
import scipy.stats as stats

In [10]:
def precision_at_q(y_true, y_hat, q=0.25):
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    pred_topk = np.argsort(y_hat)[:k]

    return len(set(true_pos) & set(pred_topk)) / k

def ndcg_at_q(y_true, y_hat, q=0.25):
    "Normalized discounted cumulative gain"
    y_true = np.asarray(y_true)
    y_hat = np.asarray(y_hat)

    thr = np.quantile(y_true, q)
    true_pos = np.where(y_true <= thr)[0]
    k = len(true_pos)

    # relevance: higher is better
    rel = -y_true
    
    # predicted ranking
    order = np.argsort(y_hat)
    rel_pred = rel[order][:k]
    
    discounts = 1 / np.log2(np.arange(2, k + 2)) # rank weight
    dcg = np.sum((2 ** rel_pred - 1) * discounts)

    # ideal ranking
    ideal_order = np.argsort(y_true)
    rel_ideal = rel[ideal_order][:k]
    idcg = np.sum((2 ** rel_ideal - 1) * discounts)

    return dcg / idcg if idcg > 0 else 0.0

In [11]:

cancer_type = "pancancer" # pancancer , solid_tumors
experiment = "NBS_cells"

per_drug_pcc_cv = {}
per_line_pcc_cv = {}

per_drug_precision_cv = {}
per_cell_precision_cv = {}

per_drug_ndcg_cv = {}
per_cell_ndcg_cv = {}

for n in range(10):
    fold_name = f"fold_{n}"
    #################################################
    # TRAIN STATS
    #################################################

    train_idx = pd.read_csv(
        f"../../cross_validation/{cancer_type}/{experiment}/fold_{n}/train_set.csv",
        index_col=0
    )
    
    train_pairs = train_idx[["DRUG_NAME", "CELL_LINE_NAME", "LN_IC50"]]
    train_pairs = train_pairs.rename(columns={"LN_IC50":"IC50"})

    train_gex = gdsc_gex.loc[train_idx.CELL_LINE_NAME]

    ########################################
    #
    # SUBSETTING TO PREDICTING SETS
    #
    ########################################
    gex = pdo_gex.loc[pdo_dr.Line,:]
    gex_mean = train_gex.mean(0)
    gex_std = train_gex.std(0)
    
    sm = smiles.loc[pdo_dr.Drug, :]

    pdo_dr["Y_TRUE"] = pdo_dr.LogIC50

    print("Entering cell feature extraction for Training")

    GeneSet_List=[]
    GeneSetFile='RawFile/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt'
    with open(GeneSetFile) as f:
        reader = csv.reader(f)
        data = list(list(rec) for rec in csv.reader(f, delimiter='\t')) #reads csv into a list of lists
        for row in data:
            GeneSet_List.append(row)

    GeneSet_Dic={}
    for GeneSet in GeneSet_List:
        GeneSet_Dic[GeneSet[0]]=GeneSet[2:]
    
    GeneSet_Dic_withoutNA={}
    for GeneSet in GeneSet_Dic:
        GeneSet_Dic_withoutNA[GeneSet] = gdsc_gex.columns.intersection(GeneSet_Dic[GeneSet]).to_list()


    pathway_name = list(GeneSet_Dic_withoutNA.keys())
    ### custom
    cellline_input = [
        pdo_gex[GeneSet_Dic_withoutNA[path]]  # Subset all cell lines at once for this gene set
        for path in pathway_name
    ]


    pathway_data = [
        cell_lines.loc[pdo_dr.Line].to_numpy()
        for cell_lines in cellline_input
    ]

    # Combine all inputs
    X = pathway_data + [sm.values]
    X = tuple(a.astype('float32', copy=False) for a in X)


    
    print("Entering model loading and prediction")

    model_path = f"models_pancancer_NBS_cells/model_{n}.keras"
    model=load_model(model_path,compile=False)

    result=model.predict(X)
    result=np.ravel(result).astype(np.float64)
    pdo_dr["Y_HAT"] = result

    # Computing metrics
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(stats.spearmanr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_drug_pcc_cv[fold_name] = [per_drug_pcc["PCC"].median()]

    per_line_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(stats.spearmanr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())

    per_line_pcc_cv[fold_name] = [per_line_pcc["PCC"].median()]


    ##############################
    # PRECISION
    ##############################
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_drug_precision_cv[fold_name] = [per_drug_pcc["precision@q25"].median()]

    per_cell_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(precision_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["precision@q25"]))
                    # .reset_index()
                   )

    per_cell_precision_cv[fold_name] = [per_cell_pcc["precision@q25"].median()]

    ##############################
    # NDCG
    ##############################
    per_drug_pcc = (pdo_dr.groupby("Drug")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Drug")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_drug_ndcg_cv[fold_name] = [per_drug_pcc["ndcg@q25"].median()]

    per_cell_pcc = (pdo_dr.groupby("Line")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Line")
                    .apply(lambda x: pd.Series(ndcg_at_q(x["Y_TRUE"], x["Y_HAT"]), index=["ndcg@q25"]))
                    # .reset_index()
                   )

    per_cell_ndcg_cv[fold_name] = [per_cell_pcc["ndcg@q25"].median()]

    

Entering cell feature extraction for Training
Entering model loading and prediction


/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


64/65 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step

/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


65/65 ━━━━━━━━━━━━━━━━━━━━ 59s 505ms/step


...

/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


64/65 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step

/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


65/65 ━━━━━━━━━━━━━━━━━━━━ 60s 531ms/step


/tmp/ipykernel_430959/3650754316.py:93: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.spearmanr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_430959/3650754316.py:101: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.spearmanr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_430959/3650754316.py:113: Depr

In [12]:
per_drug_pcc_cv = pd.DataFrame(per_drug_pcc_cv)
per_drug_pcc_cv.index = ["HiDRA"]
per_drug_pcc_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.019642,0.135317,-0.051754,-0.063158,0.00087,-0.1,0.060448,0.12588,0.0,0.132864


In [20]:
M = np.median(per_drug_pcc_cv)
sem = stats.sem(per_drug_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.01025564197373454, low=-0.05088851772399222, high=0.0713998016714613


In [21]:
per_line_pcc_cv = pd.DataFrame(per_line_pcc_cv)
per_line_pcc_cv.index = ["HiDRA"]
per_line_pcc_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.311871,0.591672,0.587438,0.583029,0.629429,0.616237,0.527006,0.641466,0.628344,0.556484


In [22]:
M = np.median(per_line_pcc_cv)
sem = stats.sem(per_line_pcc_cv.to_numpy().flatten())
ci = stats.t.interval(0.95, df=9, loc=M, scale=sem)
print(f"Median={M}, low={ci[0]}, high={ci[1]}")

Median=0.5895547196220803, low=0.5205627805717917, high=0.6585466586723688


In [13]:
per_drug_pcc_cv.to_csv(
    f"predictions_pancancer_NBS_cells/pdo_fixed-drug_CV.csv"
)

In [14]:
per_line_pcc_cv.to_csv(
    f"predictions_pancancer_NBS_cells/pdo_fixed-line_CV.csv"
)

In [13]:
per_drug_precision_cv = pd.DataFrame(per_drug_precision_cv)
per_drug_precision_cv.index = ["HiDRA"]
per_drug_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.4,0.333333,0.25,0.25,0.333333,0.275,0.333333,0.333333,0.333333,0.4


In [16]:
per_drug_precision_cv.to_csv(
    "predictions_pancancer_NBS_cells/precision_pdo_fixed-drug.csv"
)

In [17]:
per_drug_ndcg_cv = pd.DataFrame(per_drug_ndcg_cv)
per_drug_ndcg_cv.index = ["HiDRA"]
per_drug_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
per_drug_ndcg_cv.to_csv(
    "predictions_pancancer_NBS_cells/ndcg_pdo_fixed-drug.csv"
)

In [20]:
per_cell_precision_cv = pd.DataFrame(per_cell_precision_cv)
per_cell_precision_cv.index = ["HiDRA"]
per_cell_precision_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.444444,0.555556,0.555556,0.555556,0.611111,0.611111,0.555556,0.555556,0.611111,0.444444


In [22]:
per_cell_precision_cv.to_csv(
    "predictions_pancancer_NBS_cells/precision_pdo_fixed-line.csv"
)

In [23]:
per_cell_ndcg_cv = pd.DataFrame(per_cell_ndcg_cv)
per_cell_ndcg_cv.index = ["HiDRA"]
per_cell_ndcg_cv

,fold_0,fold_1,fold_2,fold_3,fold_4,fold_5,fold_6,fold_7,fold_8,fold_9
HiDRA,0.0,0.220812,0.237623,0.20754,0.152446,0.253367,0.305846,0.253533,0.268474,0.106988


In [25]:
per_cell_ndcg_cv.to_csv(
    "predictions_pancancer_NBS_cells/ndcg_pdo_fixed-line.csv"
)

In [15]:

cancer_type = "pancancer" # pancancer , solid_tumors
experiment = "NBS_cells"

per_cancer_pcc_cv = {}
per_lab_pcc_cv = {}

for n in range(10):
    fold_name = f"fold_{n}"
    #################################################
    # TRAIN STATS
    #################################################

    train_idx = pd.read_csv(
        f"../../cross_validation/{cancer_type}/{experiment}/fold_{n}/train_set.csv",
        index_col=0
    )
    
    train_pairs = train_idx[["DRUG_NAME", "CELL_LINE_NAME", "LN_IC50"]]
    train_pairs = train_pairs.rename(columns={"LN_IC50":"IC50"})

    train_gex = gdsc_gex.loc[train_idx.CELL_LINE_NAME]

    ########################################
    #
    # SUBSETTING TO PREDICTING SETS
    #
    ########################################
    gex = pdo_gex.loc[pdo_dr.Line,:]
    gex_mean = train_gex.mean(0)
    gex_std = train_gex.std(0)
    
    sm = smiles.loc[pdo_dr.Drug, :]

    pdo_dr["Y_TRUE"] = pdo_dr.LogIC50

    print("Entering cell feature extraction for Training")

    GeneSet_List=[]
    GeneSetFile='RawFile/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt'
    with open(GeneSetFile) as f:
        reader = csv.reader(f)
        data = list(list(rec) for rec in csv.reader(f, delimiter='\t')) #reads csv into a list of lists
        for row in data:
            GeneSet_List.append(row)

    GeneSet_Dic={}
    for GeneSet in GeneSet_List:
        GeneSet_Dic[GeneSet[0]]=GeneSet[2:]
    
    GeneSet_Dic_withoutNA={}
    for GeneSet in GeneSet_Dic:
        GeneSet_Dic_withoutNA[GeneSet] = gdsc_gex.columns.intersection(GeneSet_Dic[GeneSet]).to_list()


    pathway_name = list(GeneSet_Dic_withoutNA.keys())
    ### custom
    cellline_input = [
        pdo_gex[GeneSet_Dic_withoutNA[path]]  # Subset all cell lines at once for this gene set
        for path in pathway_name
    ]


    pathway_data = [
        cell_lines.loc[pdo_dr.Line].to_numpy()
        for cell_lines in cellline_input
    ]

    # Combine all inputs
    X = pathway_data + [sm.values]
    X = tuple(a.astype('float32', copy=False) for a in X)


    
    print("Entering model loading and prediction")

    model_path = f"models_pancancer_NBS_cells/model_{n}.keras"
    model=load_model(model_path,compile=False)

    result=model.predict(X)
    result=np.ravel(result).astype(np.float64)
    pdo_dr["Y_HAT"] = result

    cancer_type = (
        pdo_dr[["Line","TCGA_DESC"]]
        .drop_duplicates()
    )

    labs = (
        pdo_dr[["Line", "Lab"]]
        .drop_duplicates()
    )


    # Computing metrics
    per_cancer_pcc = (pdo_dr.groupby("TCGA_DESC")
                    .filter(lambda x: len(x) >=2)
                    .groupby("TCGA_DESC")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_cancer_pcc = (per_cancer_pcc
                      .merge(cancer_type, on="TCGA_DESC", how="left")
                      .dropna(subset=["TCGA_DESC"])
                      .groupby("TCGA_DESC")["PCC"]
                      .median()
                     )

    per_cancer_pcc_cv[fold_name] = per_cancer_pcc

    per_lab_pcc = (pdo_dr.groupby("Lab")
                    .filter(lambda x: len(x) >=2)
                    .groupby("Lab")
                    .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
                    .reset_index())
    per_lab_pcc = (per_lab_pcc
                   .merge(labs, on="Lab", how="left")
                   .dropna(subset=["Lab"])
                   .groupby("Lab")["PCC"]
                   .median()
                   )

    per_lab_pcc_cv[fold_name] = per_lab_pcc  

Entering cell feature extraction for Training
Entering model loading and prediction


/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


62/65 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step

/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


65/65 ━━━━━━━━━━━━━━━━━━━━ 52s 423ms/step


...

/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


63/65 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

/home/chp14eu/drug_repurposing/benchmarking/hidra/venv_hidra/lib/python3.11/site-packages/keras/src/ops/nn.py:908: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


65/65 ━━━━━━━━━━━━━━━━━━━━ 55s 470ms/step


/tmp/ipykernel_3713181/3205080491.py:98: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))
/tmp/ipykernel_3713181/3205080491.py:112: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series(stats.pearsonr(x["Y_TRUE"], x["Y_HAT"]), index=["PCC", "p_value"]))


In [16]:
cancer_type = (pd.DataFrame(per_cancer_pcc_cv)
               .groupby(level=0)
               .median()
               .median(axis=1)
               .rename_axis("")
               .T
)
cancer_type.name = "HiDRA"

cancer_type


BLCA    0.166064
COAD    0.618579
HNSC    0.285354
PDAC    0.737829
Name: HiDRA, dtype: float64

In [17]:
cancer_type.to_csv(
    "predictions_pancancer_NBS_cells/pdo_fixed-cancer_CV.csv"
)